# Run particle analysis

One full run: move to (x, y) → tuning protocol (stage z, C1, A1, … step by step) → HAADF overview → find particles → EDS on each particle.

**How to use**
1. Start the instrument side:
   - twin: `uv run python scripts/twin.py start`
   - real microscope: its asyncroscopy MCP server must be running; set `STEM_REAL_MCP_URL` in `.env`.
   - set the scan size, dwell time and EDS exposure by hand (they are not reachable through MCP yet).
2. Edit **Settings** below.
3. Run all cells. Results land in `runs/<time>_<tag>/` (`summary.json`, `summary.png`, tuning plots, spectra).

## Settings

In [ ]:
SETTINGS = {
    "instrument": "twin",                 # "twin" or "real"
    "stage_xy_m": (5e-9, 0.0),       # stage x, y to analyse (m)
    "tuning_fov_m": 40e-9,            # FOV of the overview (notebook used 186 nm)
    "protocol": [                     # tuning steps, in order (required)
        {"name": "z",  "tune": "z"},
        {"name": "C1", "tune": "C1", "fov_m": 40e-9},
        {"name": "A1", "tune": "A1", "fov_m": 40e-9},
    ],
    # a step may also set "metric", "step_m", "max_offset_m", "max_iterations",
    # "min_step_m" (smallest move the stage / corrector can make; z: >= stage tolerance)
    # "method": "grid" (z / C1: grid +-max_offset_m, normal-curve fit, image at the peak),
    # "rounds" (grid: zoom-in rounds, default 1)
    # (defaults per tune in stem_agent/config.py TUNING)
    "run_tag": "particles",

    # optional overrides (defaults in stem_agent/config.py):
    # "sam": {"min_visible": 0.5, "min_diameter_nm": 0.0, "edge_px": 1},   # SAM2 particles
    # "max_particles": 20,              # EDS points
    # "detector": "haadf",
    # "blank_beam_at_end": False,
}

## The graph
Solid: the main path. Red dashed: jump to `status_end` when a node fails or no particles are found. `status_end` and `save` always run, so every run leaves a record.

In [ ]:
%matplotlib inline
import json
from pathlib import Path

import numpy as np
from IPython.display import Image, display

from stem_agent.graphs.particle_graph import build_graph, run
from stem_agent.utils.plots import plot_graph

plot_graph(build_graph())

## Run
Prints each node's progress; `== <node> done` marks the end of a node.

In [ ]:
state = await run(SETTINGS)

## Results

In [ ]:
run_dir = Path(state["run"]["run_dir"])
print(f"run_status: {state['run_status']}")
print(f"errors:     {state.get('errors') or 'none'}")
print(f"folder:     {run_dir}")
for name, result in state.get("tuning", {}).items():
    unit, unit_name = (1e-6, "um") if result["tune"] == "z" else (1e-9, "nm")
    best = [round(v / unit, 3) for v in result["best_offset_m"]]
    print(f"tune {name:<6} best {best} {unit_name}, {result['settings']['metric']} {result['best_score']:.4g}, "
          f"{result['n_images']} images, {result['stop_reason']}")
if (run_dir / "summary.png").exists():
    display(Image(run_dir / "summary.png"))

### Particles and spectra

In [ ]:
spectra = {r["particle_id"]: r for r in state.get("spectra") or []}
print(f"{'id':>3} {'aim (row, col)':>16} {'area px':>8}  spectrum")
for p in state.get("particles") or []:
    record = spectra.get(p["id"])
    if record is None:
        text = "not measured (cap)"
    else:
        values = np.load(record["npy_path"])
        if record.get("elements"):
            top = sorted(zip(values, record["elements"]), reverse=True)[:3]
            text = ", ".join(f"{name} {value:.2f}" for value, name in top)
        else:
            text = f"{values.size} channels, total counts {values.sum():.0f}"
    print(f"{p['id']:>3} {f'({p["row"]:.0f}, {p["col"]:.0f})':>16} {p.get('area_px', 0):>8}  {text}")

### Tuning plots

In [ ]:
for result in state.get("tuning", {}).values():
    display(Image(result["figure_path"]))